### Event Extract from Midi File

In [2]:
import mido
import numpy as np


def get_eventlist(data_file):
    ON = 1
    OFF = 0
    CC = 2

    midi = mido.MidiFile(data_file)

    current_time = 0
    eventlist = []
    cc = False
    for msg in midi: 
        current_time += msg.time

         # NOTE ON CASE
        if msg.type is 'note_on' and msg.velocity > 0:
            event = [current_time, ON, msg.note, msg.velocity]
            eventlist.append(event)

         # NOTE OFF CASE        
        elif msg.type is 'note_off' or (msg.type is 'note_on' and msg.velocity == 0):
            event = [current_time, OFF, msg.note, msg.velocity]
            eventlist.append(event)
            
#         if msg.type is 'control_change':
            
#             if msg.control != 64:
#                 continue
            
#             if cc == False and msg.value > 0:
#                 cc = True
#                 event = [current_time, CC, 0, 1]
#                 eventlist.append(event)
                
#             elif cc == True and msg.value == 0:
#                 cc = False
#                 event = [current_time, CC, 0, 0]
#                 eventlist.append(event)
                
    eventlist = np.array(eventlist)
    return eventlist

### Midifile to EventListfile

In [13]:
from tqdm import tqdm_notebook as tqdm
import os
from os import listdir
from os.path import isfile, join

old_dir = 'original_dataset'
dataset_dir = 'dataset_after_preprocess'

if not os.path.exists(dataset_dir):
    os.makedirs(dataset_dir)
    
if not os.path.exists(str(dataset_dir)+'/train'):
    os.makedirs(str(dataset_dir)+'/train')

if not os.path.exists(str(dataset_dir)+'/test'):
    os.makedirs(str(dataset_dir)+'/test')
    
data_train_files = []
data_train_files += [join('original_dataset/train/', f)for f in listdir('original_dataset/train') if isfile(join('original_dataset/train', f)) if 'midi' in f]
for i in tqdm(range(len(data_train_files))):
    eventlist = get_eventlist(data_train_files[i]) 
#     print(eventlist)
    save_file = dataset_dir + '/train/' + str(i)
    data = {'eventlist': eventlist}
    np.savez(save_file, **data, allow_pickle=False)
  

data_test_files = []  
data_test_files += [join('original_dataset/test/', f)for f in listdir('original_dataset/test') if isfile(join('original_dataset/test', f)) if 'midi' in f]
for i in tqdm(range(len(data_test_files))):
    eventlist = get_eventlist(data_test_files[i])    
    save_file = dataset_dir + '/test/' + str(i)
    data = {'eventlist': eventlist}
    np.savez(save_file, **data, allow_pickle=False)